In [13]:
import pandas as pd
from google.colab import drive
import os

# 1. 掛載雲端硬碟
drive.mount('/content/drive')

# 2. 設定檔案路徑 (請根據你雲端硬碟的實際路徑調整)
folder_path = '/content/drive/MyDrive/金融資料/'
input_file = os.path.join(folder_path, '2022加權指數.csv')
output_file = os.path.join(folder_path, '2022加權指數_更新版.csv')

# 3. 讀取資料
# 根據資料來源 ，編碼通常為 utf-8 或 cp950
try:
    df = pd.read_csv(input_file, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(input_file, encoding='cp950')

# 4. 轉換日期格式並新增 FILE 欄位
# 資料來源  的日期格式為 '20220103' (YYYYMMDD)
# 轉換目標格式為 'OptionsDaily_YYYY_MM_DD.csv'

def format_filename(date_val):
    date_str = str(date_val)
    year = date_str[:4]
    month = date_str[4:6]
    day = date_str[6:8]
    return f"OptionsDaily_{year}_{month}_{day}.csv"

# 套用命名規則
df['FILE'] = df['年月日'].apply(format_filename)

# 5. 儲存檔案
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"處理完成！檔案已儲存至: {output_file}")
print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
處理完成！檔案已儲存至: /content/drive/MyDrive/金融資料/2022加權指數_更新版.csv
         證券代碼       年月日    收盤價(元)                         FILE
0  Y9999 加權指數  20220103  18270.51  OptionsDaily_2022_01_03.csv
1  Y9999 加權指數  20220104  18526.35  OptionsDaily_2022_01_04.csv
2  Y9999 加權指數  20220105  18499.96  OptionsDaily_2022_01_05.csv
3  Y9999 加權指數  20220106  18367.92  OptionsDaily_2022_01_06.csv
4  Y9999 加權指數  20220107  18169.76  OptionsDaily_2022_01_07.csv


In [14]:
import pandas as pd
from google.colab import drive
import os

# 1. 掛載雲端硬碟
drive.mount('/content/drive')

# 2. 設定檔案路徑 (請根據實際路徑調整)
folder_path = '/content/drive/MyDrive/金融資料/'
index_file = os.path.join(folder_path, '2022加權指數_更新版.csv')
expiry_file = os.path.join(folder_path, '2022資料 - 工作表1.csv')
output_file = os.path.join(folder_path, '2022加權指數_最終版.csv')

# 3. 讀取資料
df_index = pd.read_csv(index_file)
df_expiry = pd.read_csv(expiry_file)

# 清理欄位名稱 (移除換行符號)
df_expiry.columns = [c.replace('\n', '') for c in df_expiry.columns]

# 4. 篩選「月選擇權」並轉換日期格式
# 排除契約月份中包含 'W' 的資料
df_monthly = df_expiry[~df_expiry['契約月份'].astype(str).str.contains('W')].copy()

# 轉換為日期格式以便計算
df_monthly['最後結算日'] = pd.to_datetime(df_monthly['最後結算日'])
df_index['年月日_dt'] = pd.to_datetime(df_index['年月日'].astype(str), format='%Y%m%d')

# 確保日期排序由舊到新
df_monthly = df_monthly.sort_values('最後結算日')

# 5. 定義邏輯：找出距離交易日 >= 1 天的最近月契約
def get_nearest_contract(row):
    trading_date = row['年月日_dt']
    # 篩選出大於或等於交易日 + 1 天的到期日
    future_expiries = df_monthly[df_monthly['最後結算日'] >= trading_date + pd.Timedelta(days=1)]

    if not future_expiries.empty:
        # 取得最接近的一個 (第一列)
        nearest = future_expiries.iloc[0]
        contract = nearest['契約月份']
        expiry_date = nearest['最後結算日']
        # 計算剩餘天數 (Maturity)
        maturity = (expiry_date - trading_date).days
        return pd.Series([contract, expiry_date.strftime('%Y/%m/%d'), maturity])
    else:
        return pd.Series([None, None, None])

# 6. 執行計算並新增欄位
df_index[['contract', 'contractExpirydate', 'maturity']] = df_index.apply(get_nearest_contract, axis=1)

# 移除輔助用的日期欄位
df_index = df_index.drop(columns=['年月日_dt'])

# 7. 儲存結果
df_index.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"處理完成！結果已儲存至: {output_file}")
print(df_index[['年月日', 'contract', 'contractExpirydate', 'maturity']].head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
處理完成！結果已儲存至: /content/drive/MyDrive/金融資料/2022加權指數_最終版.csv
        年月日 contract contractExpirydate  maturity
0  20220103   202201         2022/01/19        16
1  20220104   202201         2022/01/19        15
2  20220105   202201         2022/01/19        14
3  20220106   202201         2022/01/19        13
4  20220107   202201         2022/01/19        12
5  20220110   202201         2022/01/19         9
6  20220111   202201         2022/01/19         8
7  20220112   202201         2022/01/19         7
8  20220113   202201         2022/01/19         6
9  20220114   202201         2022/01/19         5


In [16]:
import pandas as pd
from google.colab import drive
import os

# 1. 掛載雲端硬碟
drive.mount('/content/drive')

# 2. 設定檔案路徑
folder_path = '/content/drive/MyDrive/金融資料/'
input_file = os.path.join(folder_path, '2022加權指數_最終版.csv')
output_file = os.path.join(folder_path, '2022加權指數_含RF_修正版.csv')

# 3. 讀取資料
df = pd.read_csv(input_file)
df['date_int'] = df['年月日'].astype(int)

# 4. 修正後的 2022 年 RF 邏輯 (調整日當天維持前一期利率)
def get_rf_rate_corrected(date):
    # 3/18 當天仍為 0.84%
    if date <= 20220318:
        return 0.00840
    # 6/17 當天仍為 1.09%
    elif 20220318 < date <= 20220617:
        return 0.01090
    # 9/23 當天仍為 1.215%
    elif 20220617 < date <= 20220923:
        return 0.01215
    # 12/16 當天仍為 1.340%
    elif 20220923 < date <= 20221216:
        return 0.01340
    # 12/16 之後才調整為 1.465%
    else:
        return 0.01465

# 5. 更新 RF 欄位
df['RF'] = df['date_int'].apply(get_rf_rate_corrected)

# 移除輔助欄位並儲存
df = df.drop(columns=['date_int'])
df.to_csv(output_file, index=False, encoding='utf-8-sig')

# 驗證關鍵日期的利率是否正確
check_dates = [20220318, 20220321, 20220617, 20220620, 20220923, 20220926, 20221216, 20221219]
print("--- 關鍵日期利率驗證 ---")
print(df[df['年月日'].isin(check_dates)][['年月日', 'RF']].sort_values('年月日'))
print(f"\n修正後的檔案已儲存至: {output_file}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- 關鍵日期利率驗證 ---
          年月日       RF
46   20220318  0.00840
47   20220321  0.01090
107  20220617  0.01090
108  20220620  0.01215
176  20220923  0.01215
177  20220926  0.01340
235  20221216  0.01340
236  20221219  0.01465

修正後的檔案已儲存至: /content/drive/MyDrive/金融資料/2022加權指數_含RF_修正版.csv
